In [10]:
import cracknuts as cn

In [306]:
o1 = cn.cracker_o1('192.168.0.23')
o1.connect(force_update_bin=True, force_write_default_config=True)
# o1.connect(force_update_bin=True)
# o1.connect()

In [ ]:
o1.get_hardware_model()

In [ ]:
o1.get_firmware_info()

In [ ]:
# 波形生成

In [ ]:
# 设置波形
o1.set_waveform([1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,
                 0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0], wait=1, delay=1, repeat=10000)

In [ ]:
o1.set_waveform([2,2,2,2,2,0,0,0,0,0], wait=1, delay=1, repeat=10000)

In [ ]:
o1.set_waveform([1,1,1,1,1,0.5,0,0,0,0], wait=1, delay=1, repeat=10000)

In [ ]:
o1.set_waveform([0,0,0,0,0,0,0,0,0,0], wait=1, delay=1, repeat=10000)

In [ ]:
o1.set_waveform([0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0], wait=1, delay=1, repeat=10000)

In [ ]:
o1.set_waveform([2,2,2,2,2], wait=1, delay=1, repeat=10000)

In [ ]:
# 使用
o1.disable_waveform()

In [ ]:
# 禁止
o1.enable_waveform()

In [ ]:
o1.set_pwm('gp29', 10_000, 0.5)

In [ ]:
o1.set_pwm('gp28', 10_000, 0.5)

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x1810, data=4)
o1.register_write(base_address=0x43c10000, offset=0x1814, data=4)

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x180c, data=1)

In [ ]:
v = o1.register_read(base_address=0x43c10000, offset=0x180c)[1]
print(int.from_bytes(v))

In [ ]:
# 获取电压
print(o1.get_voltage_a0())
print(o1.get_voltage_a1())

##### 无法获取电压，输入 j50 的 a0 和 a1

In [ ]:
o1.register_read(base_address=0x43C10000, offset=0x1890)

In [ ]:
o1.register_read(base_address=0x43C10000, offset=0x189c)

In [ ]:
o1.register_read(base_address=0x43C10000, offset=0x1800)

In [ ]:
o1.glitch_clock_force()
o1.register_read(base_address=0x43c10000, offset=0x1890)

In [ ]:
# 189c  
o1.glitch_clock_force()
o1.register_read(base_address=0x43c10000, offset=0x189c)

In [ ]:
acq = cn.simple_acq(cracker=o1)
cn.show_panel(acq)

In [ ]:
o1.osc_is_triggered()

In [ ]:
# 0x43C20000
o1.register_read(base_address=0x43C20000, offset=0x0)

In [ ]:
o1.uart_enable()

In [ ]:
o1.register_read(base_address=0x43C10000, offset=0x44)

In [ ]:
o1.register_write(base_address=0x43C10000, offset=0x44, data=1)

In [ ]:
o1.osc_force()

In [ ]:
o1.osc_single()

In [ ]:
o1.osc_sample_length(1000)
o1.osc_trigger_source('A')
o1.osc_trigger_mode('E')

In [ ]:
o1.reconnect()

In [ ]:
o1.register_read(base_address=0x43c10000, offset=0x189c)

In [ ]:
o1.register_read(base_address=0x43C2_0000, offset=0x00)

In [ ]:
o1.osc_is_triggered()

In [ ]:
o1.get_firmware_info()

In [ ]:
o1.register_read(base_address=0x43C2_0000, offset=0x00)

In [ ]:
# 显示屏幕

In [ ]:
# 显示文字

import struct

t = 0
x = 0
y = 7
c = 'KKKK'

payload = struct.pack('>Bii', t, x, y)
payload += c.encode('utf8')

print(payload.hex(' '))

o1.send_with_command(command=0x400, payload=payload)

In [ ]:
from PIL import Image
import numpy as np

def resize_to_rgb888_64x64_crop(image_path: str) -> np.ndarray:
    """
    读取图片，保持比例裁剪中心区域，缩放为 64x64，转换为 RGB888 数组。
    
    策略：
    1. 找到短边，以中心为基准裁剪出正方形 (Short Side Crop)。
    2. 将正方形缩放至 64x64。
    3. 确保输出为 uint8 RGB 格式。
    """
    TARGET_SIZE = 64
    
    try:
        with Image.open(image_path) as img:
            # 1. 转 RGB (处理透明通道等)
            rgb_img = img.convert('RGB')
            
            width, height = rgb_img.size
            
            # 2. 计算裁剪区域 (Center Crop)
            # 找出短边长度
            min_dim = min(width, height)
            
            # 计算中心点
            center_x = width // 2
            center_y = height // 2
            
            # 计算裁剪框的左上角和右下角坐标
            left = center_x - (min_dim // 2)
            top = center_y - (min_dim // 2)
            right = left + min_dim
            bottom = top + min_dim
            
            # 执行裁剪 (得到正方形图片)
            square_img = rgb_img.crop((left, top, right, bottom))
            
            # 3. 缩放至 64x64
            # 因为已经是正方形了，直接 resize 不会变形
            resized_img = square_img.resize((TARGET_SIZE, TARGET_SIZE), Image.Resampling.LANCZOS)
            
            # 4. 转 NumPy 数组
            rgb_array = np.array(resized_img, dtype=np.uint8)
            
            # 验证
            assert rgb_array.shape == (64, 64, 3), f"形状错误: {rgb_array.shape}"
            
            return rgb_array
            
    except FileNotFoundError:
        print(f"错误：找不到文件 '{image_path}'")
        return None
    except Exception as e:
        print(f"发生错误: {e}")
        return None

In [ ]:
image = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 2, 2, 29, 9, 9, 8, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 1, 1, 26, 8, 8, 106, 25, 25, 157, 45, 45, 43, 13, 13, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 1, 1, 26, 7, 7, 83, 26, 26, 159, 48, 48, 207, 61, 61, 210, 62, 62, 104, 30, 30, 6, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 14, 4, 4, 63, 17, 17, 158, 41, 41, 204, 61, 61, 217, 65, 65, 219, 65, 65, 218, 65, 65, 163, 48, 48, 23, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 13, 2, 2, 57, 17, 17, 131, 40, 40, 192, 57, 57, 216, 64, 64, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 198, 59, 59, 60, 18, 18, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 1, 1, 27, 8, 8, 66, 20, 20, 25, 8, 8, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 5, 4, 0, 6, 5, 0, 6, 5, 0, 6, 5, 0, 6, 5, 0, 5, 3, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 30, 22, 0, 69, 50, 0, 48, 35, 0, 9, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 32, 10, 10, 116, 31, 31, 193, 55, 55, 215, 64, 64, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 214, 63, 63, 124, 37, 37, 11, 3, 3, 0, 0, 0, 0, 0, 0, 4, 1, 1, 28, 8, 8, 87, 27, 27, 162, 48, 48, 197, 59, 59, 99, 34, 34, 5, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 11, 8, 0, 33, 24, 0, 61, 44, 0, 102, 73, 0, 117, 84, 0, 117, 85, 0, 117, 85, 0, 117, 84, 0, 101, 73, 0, 59, 43, 0, 30, 22, 0, 10, 7, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 34, 24, 0, 151, 109, 0, 192, 139, 0, 173, 125, 0, 68, 49, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 10, 3, 3, 127, 40, 40, 209, 62, 62, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 218, 65, 65, 176, 52, 52, 32, 9, 9, 2, 1, 1, 15, 5, 5, 68, 18, 18, 163, 43, 43, 206, 62, 62, 217, 65, 65, 217, 65, 65, 152, 46, 46, 18, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 0, 23, 17, 0, 73, 53, 0, 124, 90, 0, 167, 121, 0, 190, 137, 0, 199, 144, 0, 202, 146, 0, 202, 146, 0, 202, 146, 0, 202, 146, 0, 199, 144, 0, 189, 137, 0, 165, 119, 0, 117, 85, 0, 49, 36, 0, 11, 8, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 51, 37, 0, 187, 135, 0, 205, 148, 0, 203, 147, 0, 158, 114, 0, 32, 23, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 2, 2, 103, 34, 34, 210, 63, 63, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 216, 64, 64, 196, 58, 58, 133, 39, 39, 42, 10, 10, 61, 19, 19, 136, 41, 41, 195, 58, 58, 217, 64, 64, 219, 65, 65, 219, 65, 65, 219, 65, 65, 191, 56, 56, 50, 15, 15, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 17, 13, 0, 80, 63, 0, 148, 108, 0, 191, 138, 0, 202, 146, 0, 204, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 204, 148, 0, 200, 145, 0, 177, 128, 0, 114, 82, 0, 33, 23, 0, 4, 3, 0, 1, 0, 0, 35, 28, 0, 165, 121, 0, 204, 147, 0, 205, 148, 0, 193, 139, 0, 71, 51, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 40, 12, 12, 185, 55, 55, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 218, 65, 65, 207, 59, 59, 151, 42, 42, 65, 19, 19, 31, 9, 9, 119, 31, 31, 195, 56, 56, 215, 64, 64, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 213, 63, 63, 120, 36, 36, 8, 3, 3, 0, 0, 0, 0, 0, 0, 2, 2, 0, 31, 21, 0, 111, 80, 0, 190, 140, 0, 203, 147, 0, 205, 148, 0, 205, 148, 0, 204, 147, 0, 201, 145, 0, 191, 138, 0, 188, 136, 0, 181, 134, 0, 172, 124, 0, 188, 135, 0, 192, 138, 0, 201, 145, 0, 204, 148, 0, 205, 148, 0, 205, 148, 0, 197, 142, 0, 155, 112, 0, 70, 54, 0, 13, 11, 0, 7, 6, 0, 103, 75, 0, 200, 144, 0, 205, 148, 0, 196, 142, 0, 76, 55, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 13, 4, 4, 132, 40, 40, 215, 64, 64, 219, 65, 65, 219, 65, 65, 218, 65, 65, 207, 61, 61, 166, 49, 49, 93, 25, 25, 28, 5, 5, 3, 1, 1, 33, 9, 9, 177, 52, 52, 218, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 218, 65, 65, 168, 50, 50, 26, 8, 8, 0, 0, 0, 2, 1, 0, 40, 29, 0, 145, 104, 0, 197, 141, 0, 205, 148, 0, 205, 148, 0, 204, 147, 0, 191, 138, 0, 161, 116, 0, 115, 82, 0, 66, 47, 0, 52, 38, 0, 45, 36, 0, 37, 26, 0, 52, 37, 0, 69, 49, 0, 119, 85, 0, 171, 123, 0, 199, 143, 0, 205, 148, 0, 205, 148, 0, 203, 147, 0, 185, 137, 0, 101, 75, 0, 36, 29, 0, 120, 87, 0, 201, 145, 0, 205, 148, 0, 196, 142, 0, 76, 55, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 67, 20, 20, 201, 60, 60, 218, 65, 65, 213, 63, 63, 185, 50, 50, 97, 27, 27, 30, 9, 9, 5, 1, 1, 0, 0, 0, 0, 0, 0, 11, 3, 3, 126, 37, 37, 214, 64, 64, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 218, 65, 65, 211, 63, 63, 163, 52, 52, 31, 10, 10, 0, 0, 0, 16, 11, 0, 135, 97, 0, 200, 145, 0, 205, 148, 0, 205, 148, 0, 202, 145, 0, 165, 117, 0, 78, 56, 0, 27, 19, 0, 8, 6, 0, 2, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 2, 1, 0, 9, 7, 0, 41, 30, 0, 116, 83, 0, 185, 133, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 194, 142, 0, 164, 123, 0, 185, 134, 0, 205, 148, 0, 205, 148, 0, 185, 134, 0, 58, 42, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 27, 8, 8, 165, 50, 50, 185, 55, 55, 121, 36, 36, 50, 12, 12, 8, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 67, 21, 21, 201, 60, 60, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 211, 63, 63, 177, 53, 53, 107, 32, 32, 39, 14, 14, 4, 1, 1, 0, 0, 0, 22, 16, 0, 161, 116, 0, 204, 148, 0, 205, 148, 0, 205, 148, 0, 195, 140, 0, 112, 83, 0, 24, 20, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 15, 10, 0, 74, 52, 0, 169, 122, 0, 201, 145, 0, 205, 148, 0, 205, 148, 0, 204, 148, 0, 205, 148, 0, 205, 148, 0, 202, 146, 0, 137, 99, 0, 18, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 2, 2, 61, 19, 19, 47, 14, 14, 11, 3, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 28, 10, 10, 173, 55, 55, 218, 65, 65, 219, 65, 65, 215, 64, 64, 195, 59, 59, 123, 40, 40, 39, 12, 12, 7, 2, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 13, 10, 0, 123, 90, 0, 194, 141, 0, 204, 148, 0, 205, 148, 0, 204, 147, 0, 192, 141, 0, 120, 89, 0, 33, 23, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 4, 0, 48, 35, 0, 139, 100, 0, 196, 141, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 185, 133, 0, 69, 49, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 3, 3, 107, 33, 33, 209, 62, 62, 194, 57, 57, 136, 40, 40, 61, 18, 18, 14, 5, 5, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 25, 20, 0, 99, 74, 0, 177, 129, 0, 202, 146, 0, 205, 148, 0, 205, 148, 0, 197, 142, 0, 154, 111, 0, 65, 50, 0, 10, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 29, 20, 0, 105, 75, 0, 186, 133, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 196, 142, 0, 140, 97, 0, 46, 32, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 46, 14, 14, 140, 36, 36, 68, 18, 18, 15, 4, 4, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 10, 7, 0, 58, 43, 0, 158, 118, 0, 199, 145, 0, 205, 148, 0, 205, 148, 0, 203, 147, 0, 180, 132, 0, 94, 69, 0, 23, 17, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 13, 9, 0, 78, 55, 0, 164, 119, 0, 201, 146, 0, 205, 148, 0, 205, 148, 0, 201, 145, 0, 158, 115, 0, 71, 55, 0, 10, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 1, 1, 15, 3, 3, 9, 2, 2, 23, 7, 7, 13, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 0, 40, 33, 0, 120, 87, 0, 185, 134, 0, 204, 147, 0, 205, 148, 0, 204, 148, 0, 193, 140, 0, 136, 100, 0, 50, 40, 0, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 4, 0, 49, 38, 0, 136, 101, 0, 196, 141, 0, 205, 148, 0, 205, 148, 0, 203, 147, 0, 181, 131, 0, 84, 60, 0, 10, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 24, 5, 5, 94, 23, 23, 159, 47, 47, 94, 29, 29, 5, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 15, 11, 0, 78, 58, 0, 174, 130, 0, 201, 146, 0, 205, 148, 0, 205, 148, 0, 201, 146, 0, 167, 122, 0, 77, 56, 0, 16, 11, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 26, 19, 0, 117, 80, 0, 183, 132, 0, 204, 147, 0, 205, 148, 0, 204, 147, 0, 181, 131, 0, 83, 60, 0, 10, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 16, 5, 5, 64, 19, 19, 145, 42, 42, 207, 59, 59, 216, 64, 64, 156, 46, 46, 23, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 6, 0, 50, 39, 0, 136, 98, 0, 192, 139, 0, 205, 148, 0, 205, 148, 0, 204, 147, 0, 188, 136, 0, 121, 87, 0, 37, 26, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 13, 9, 0, 78, 56, 0, 165, 119, 0, 201, 145, 0, 205, 148, 0, 204, 147, 0, 181, 131, 0, 81, 59, 0, 6, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 1, 1, 61, 13, 13, 141, 39, 39, 196, 58, 58, 216, 64, 64, 219, 65, 65, 219, 65, 65, 202, 57, 57, 70, 16, 16, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 21, 15, 0, 105, 79, 0, 182, 135, 0, 203, 147, 0, 205, 148, 0, 205, 148, 0, 199, 143, 0, 149, 108, 0, 62, 45, 0, 10, 7, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 4, 0, 46, 33, 0, 150, 109, 0, 201, 145, 0, 205, 148, 0, 204, 147, 0, 167, 122, 0, 46, 34, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 41, 12, 12, 177, 50, 50, 216, 63, 63, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 214, 63, 63, 121, 35, 35, 8, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 10, 0, 64, 48, 0, 150, 108, 0, 197, 142, 0, 205, 148, 0, 205, 148, 0, 202, 146, 0, 179, 129, 0, 107, 75, 0, 30, 23, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 43, 31, 0, 156, 112, 0, 203, 147, 0, 205, 148, 0, 201, 145, 0, 131, 95, 0, 19, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 37, 11, 11, 181, 54, 54, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 219, 65, 65, 214, 64, 64, 149, 44, 44, 19, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 30, 21, 0, 107, 77, 0, 184, 133, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 196, 142, 0, 153, 116, 0, 52, 39, 0, 7, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 0, 71, 50, 0, 184, 134, 0, 205, 148, 0, 205, 148, 0, 190, 138, 0, 80, 61, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 13, 4, 4, 137, 42, 42, 216, 64, 64, 219, 65, 65, 219, 65, 65, 215, 64, 64, 192, 57, 57, 131, 40, 40, 52, 16, 16, 5, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 12, 9, 0, 80, 58, 0, 162, 117, 0, 200, 144, 0, 205, 148, 0, 205, 148, 0, 202, 146, 0, 168, 123, 0, 88, 64, 0, 22, 14, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14, 11, 0, 121, 93, 0, 198, 144, 0, 205, 148, 0, 203, 147, 0, 142, 102, 0, 17, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 14, 10, 0, 22, 16, 0, 15, 11, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 3, 1, 1, 86, 30, 30, 207, 63, 63, 217, 64, 64, 203, 61, 61, 141, 42, 42, 57, 17, 17, 13, 4, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 4, 0, 42, 30, 0, 125, 90, 0, 193, 139, 0, 204, 147, 0, 205, 148, 0, 204, 147, 0, 192, 137, 0, 133, 93, 0, 36, 25, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 35, 26, 0, 164, 118, 0, 204, 147, 0, 205, 148, 0, 184, 134, 0, 61, 48, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 30, 19, 0, 128, 91, 0, 161, 116, 0, 132, 95, 0, 33, 23, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 29, 9, 9, 154, 46, 46, 157, 47, 47, 82, 25, 25, 21, 7, 7, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 21, 15, 0, 95, 68, 0, 174, 126, 0, 202, 146, 0, 205, 148, 0, 205, 148, 0, 199, 143, 0, 158, 113, 0, 73, 50, 0, 15, 9, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 0, 115, 84, 0, 201, 145, 0, 205, 148, 0, 201, 146, 0, 104, 78, 0, 6, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 76, 52, 0, 194, 139, 0, 204, 148, 0, 195, 141, 0, 82, 59, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 4, 1, 1, 37, 11, 11, 24, 7, 7, 4, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 0, 55, 40, 0, 157, 114, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 203, 147, 0, 187, 132, 0, 108, 76, 0, 26, 18, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 64, 47, 0, 189, 137, 0, 205, 148, 0, 203, 147, 0, 141, 101, 0, 18, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 78, 56, 0, 197, 142, 0, 205, 148, 0, 199, 143, 0, 88, 63, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 117, 85, 0, 202, 146, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 194, 140, 0, 144, 103, 0, 55, 42, 0, 7, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 28, 20, 0, 167, 120, 0, 205, 148, 0, 204, 148, 0, 159, 115, 0, 24, 17, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 78, 57, 0, 197, 142, 0, 205, 148, 0, 199, 143, 0, 88, 63, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 4, 0, 115, 83, 0, 202, 146, 0, 205, 148, 0, 205, 148, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 202, 146, 0, 174, 127, 0, 83, 60, 0, 19, 13, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 22, 16, 0, 162, 117, 0, 205, 148, 0, 204, 148, 0, 159, 115, 0, 24, 17, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 78, 57, 0, 197, 142, 0, 205, 148, 0, 199, 143, 0, 88, 63, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 4, 0, 115, 84, 0, 202, 146, 0, 205, 148, 0, 201, 145, 0, 176, 128, 0, 191, 138, 0, 204, 148, 0, 205, 148, 0, 204, 147, 0, 189, 137, 0, 126, 91, 0, 43, 30, 0, 4, 3, 0, 0, 0, 0, 22, 16, 0, 162, 117, 0, 205, 148, 0, 204, 148, 0, 159, 115, 0, 24, 17, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 78, 56, 0, 197, 142, 0, 205, 148, 0, 199, 143, 0, 89, 64, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 4, 0, 116, 83, 0, 202, 146, 0, 205, 148, 0, 196, 143, 0, 102, 78, 0, 94, 69, 0, 181, 134, 0, 202, 146, 0, 205, 148, 0, 205, 148, 0, 200, 144, 0, 159, 114, 0, 66, 48, 0, 12, 9, 0, 27, 20, 0, 165, 119, 0, 205, 148, 0, 204, 148, 0, 159, 115, 0, 24, 17, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 78, 54, 0, 196, 141, 0, 205, 148, 0, 201, 145, 0, 117, 83, 0, 8, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 4, 0, 115, 83, 0, 202, 146, 0, 205, 148, 0, 190, 137, 0, 57, 42, 0, 11, 9, 0, 60, 47, 0, 148, 106, 0, 195, 141, 0, 204, 148, 0, 205, 148, 0, 203, 146, 0, 183, 132, 0, 111, 81, 0, 84, 62, 0, 187, 136, 0, 205, 148, 0, 204, 147, 0, 146, 107, 0, 19, 15, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 47, 31, 0, 181, 129, 0, 205, 148, 0, 204, 147, 0, 147, 106, 0, 15, 11, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 4, 0, 116, 83, 0, 202, 146, 0, 205, 148, 0, 189, 136, 0, 54, 39, 0, 1, 1, 0, 3, 2, 0, 27, 19, 0, 103, 75, 0, 178, 129, 0, 203, 147, 0, 205, 148, 0, 205, 148, 0, 198, 143, 0, 180, 130, 0, 202, 146, 0, 205, 148, 0, 201, 146, 0, 107, 80, 0, 7, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 29, 21, 0, 170, 123, 0, 205, 148, 0, 204, 148, 0, 159, 115, 0, 25, 18, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 12, 9, 0, 130, 95, 0, 203, 147, 0, 205, 148, 0, 188, 136, 0, 53, 38, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 10, 7, 0, 71, 50, 0, 159, 114, 0, 199, 144, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 188, 137, 0, 69, 54, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 18, 13, 0, 145, 105, 0, 204, 147, 0, 205, 148, 0, 185, 134, 0, 52, 38, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 26, 19, 0, 165, 120, 0, 205, 148, 0, 205, 148, 0, 170, 123, 0, 35, 26, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 3, 0, 37, 27, 0, 119, 86, 0, 189, 136, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 202, 146, 0, 138, 100, 0, 18, 13, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 120, 89, 0, 203, 147, 0, 205, 148, 0, 198, 145, 0, 95, 75, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 45, 33, 0, 179, 129, 0, 205, 148, 0, 204, 147, 0, 151, 108, 0, 17, 12, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 16, 12, 0, 90, 65, 0, 170, 123, 0, 201, 145, 0, 204, 147, 0, 184, 133, 0, 67, 48, 0, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 0, 87, 68, 0, 196, 142, 0, 205, 148, 0, 202, 146, 0, 125, 91, 0, 10, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 0, 90, 65, 0, 197, 142, 0, 205, 148, 0, 201, 145, 0, 117, 83, 0, 9, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 50, 37, 0, 122, 88, 0, 142, 102, 0, 90, 64, 0, 15, 10, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 40, 30, 0, 174, 126, 0, 205, 148, 0, 204, 148, 0, 167, 120, 0, 32, 23, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 16, 12, 0, 141, 102, 0, 203, 147, 0, 205, 148, 0, 189, 137, 0, 66, 48, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 10, 7, 0, 13, 10, 0, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 14, 10, 0, 135, 98, 0, 203, 146, 0, 205, 148, 0, 194, 140, 0, 88, 64, 0, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 59, 43, 0, 183, 132, 0, 205, 148, 0, 204, 148, 0, 162, 117, 0, 28, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 83, 60, 0, 194, 140, 0, 205, 148, 0, 204, 147, 0, 161, 113, 0, 32, 22, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 15, 11, 0, 130, 94, 0, 202, 146, 0, 205, 148, 0, 201, 145, 0, 116, 85, 0, 9, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 31, 22, 0, 159, 115, 0, 204, 147, 0, 205, 148, 0, 196, 143, 0, 92, 71, 0, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 61, 44, 0, 183, 132, 0, 205, 148, 0, 205, 148, 0, 179, 129, 0, 52, 38, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 4, 0, 92, 66, 0, 195, 141, 0, 205, 148, 0, 204, 147, 0, 156, 113, 0, 29, 21, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 28, 20, 0, 148, 107, 0, 202, 146, 0, 205, 148, 0, 201, 145, 0, 122, 88, 0, 12, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 31, 22, 0, 159, 115, 0, 204, 147, 0, 205, 148, 0, 195, 141, 0, 99, 71, 0, 9, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 17, 15, 0, 116, 87, 0, 196, 141, 0, 205, 148, 0, 204, 148, 0, 173, 128, 0, 50, 39, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 4, 0, 91, 66, 0, 195, 141, 0, 205, 148, 0, 204, 148, 0, 176, 127, 0, 59, 42, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 79, 59, 0, 189, 139, 0, 205, 148, 0, 205, 148, 0, 194, 140, 0, 95, 69, 0, 8, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 28, 20, 0, 145, 105, 0, 202, 146, 0, 205, 148, 0, 203, 146, 0, 164, 119, 0, 51, 41, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 72, 51, 0, 175, 126, 0, 204, 147, 0, 205, 148, 0, 201, 146, 0, 138, 103, 0, 24, 18, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 2, 0, 61, 43, 0, 173, 125, 0, 204, 147, 0, 205, 148, 0, 202, 146, 0, 135, 100, 0, 26, 19, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 71, 51, 0, 174, 126, 0, 204, 147, 0, 205, 148, 0, 202, 146, 0, 160, 116, 0, 49, 37, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 84, 62, 0, 190, 138, 0, 205, 148, 0, 205, 148, 0, 196, 141, 0, 124, 90, 0, 25, 18, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 71, 52, 0, 174, 126, 0, 204, 147, 0, 205, 148, 0, 204, 147, 0, 172, 122, 0, 55, 39, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 17, 13, 0, 112, 81, 0, 193, 139, 0, 205, 148, 0, 205, 148, 0, 196, 141, 0, 130, 96, 0, 47, 38, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 18, 14, 0, 85, 62, 0, 175, 126, 0, 204, 147, 0, 205, 148, 0, 204, 147, 0, 184, 131, 0, 90, 60, 0, 8, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 19, 14, 0, 112, 81, 0, 193, 139, 0, 205, 148, 0, 205, 148, 0, 201, 146, 0, 162, 120, 0, 56, 41, 0, 8, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 0, 35, 25, 0, 118, 85, 0, 190, 138, 0, 204, 147, 0, 205, 148, 0, 202, 146, 0, 174, 123, 0, 91, 61, 0, 12, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 19, 14, 0, 112, 81, 0, 193, 139, 0, 205, 148, 0, 205, 148, 0, 203, 146, 0, 173, 124, 0, 95, 68, 0, 23, 16, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 21, 18, 0, 75, 55, 0, 155, 112, 0, 198, 143, 0, 205, 148, 0, 205, 148, 0, 202, 146, 0, 160, 115, 0, 55, 39, 0, 8, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 19, 14, 0, 113, 81, 0, 193, 141, 0, 204, 148, 0, 205, 148, 0, 204, 147, 0, 193, 139, 0, 138, 99, 0, 55, 39, 0, 13, 9, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 11, 8, 0, 50, 36, 0, 134, 101, 0, 190, 139, 0, 203, 147, 0, 205, 148, 0, 205, 148, 0, 196, 141, 0, 143, 101, 0, 51, 36, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 20, 15, 0, 95, 74, 0, 172, 125, 0, 201, 145, 0, 205, 148, 0, 205, 148, 0, 201, 145, 0, 180, 130, 0, 125, 89, 0, 55, 39, 0, 13, 9, 0, 3, 2, 0, 11, 8, 0, 49, 36, 0, 118, 85, 0, 177, 128, 0, 201, 145, 0, 205, 148, 0, 205, 148, 0, 203, 147, 0, 183, 132, 0, 114, 81, 0, 26, 18, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 6, 0, 50, 36, 0, 136, 99, 0, 191, 138, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 201, 145, 0, 180, 130, 0, 126, 91, 0, 91, 65, 0, 122, 88, 0, 177, 128, 0, 200, 144, 0, 205, 148, 0, 205, 148, 0, 204, 148, 0, 194, 140, 0, 150, 108, 0, 72, 52, 0, 13, 9, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 0, 21, 16, 0, 91, 67, 0, 166, 120, 0, 197, 142, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 202, 146, 0, 199, 144, 0, 202, 145, 0, 205, 148, 0, 205, 148, 0, 204, 148, 0, 199, 143, 0, 170, 123, 0, 99, 72, 0, 26, 19, 0, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 5, 0, 37, 27, 0, 101, 73, 0, 166, 120, 0, 197, 142, 0, 204, 147, 0, 205, 148, 0, 205, 148, 0, 205, 148, 0, 204, 147, 0, 198, 143, 0, 170, 123, 0, 108, 77, 0, 41, 30, 0, 8, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 7, 5, 0, 37, 27, 0, 101, 73, 0, 167, 120, 0, 196, 140, 0, 197, 142, 0, 191, 138, 0, 166, 118, 0, 107, 76, 0, 41, 30, 0, 8, 6, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 7, 5, 0, 38, 25, 0, 76, 52, 0, 78, 56, 0, 67, 48, 0, 30, 21, 0, 8, 6, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 1, 0, 2, 1, 0, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
p = bytes(image)

In [ ]:
# 显示图片

import struct

t = 1
x = 0
y = 0
w = 64
c = resize_to_rgb888_64x64_crop(r'D:\download\719e0de3234261040be0ad19efd47025.jpeg').tobytes()

payload = struct.pack('>BiiI', t, x, y, w)
payload += c

o1.send_with_command(command=0x400, payload=payload)

In [ ]:
resize_to_rgb888_64x64(r'D:\work\10.document\01.CrackNuts\网站相关内容\logo_2.png').flatten()

In [ ]:
#pwm
o1.register_write(base_address=0x43c10000, offset=0x1830, data=0x0006_8DB9)
o1.register_write(base_address=0x43c10000, offset=0x1834, data=0x7FFF_FFFF)

In [ ]:
#pwm
o1.register_write(base_address=0x43c10000, offset=0x1838, data=0x0006_8DB9)
o1.register_write(base_address=0x43c10000, offset=0x183c, data=0x7FFF_FFFF)

In [ ]:
o1.register_read(base_address=0x43c10000, offset=0x1e00)[1].hex()

In [ ]:
o1.register_read(base_address=0x43c10000, offset=0x1e04)[1]

In [ ]:
# a1
int.from_bytes(o1.register_read(base_address=0x43c10000, offset=0x1e40)[1])/16/4096*3.33

In [ ]:
# a0
int.from_bytes(o1.register_read(base_address=0x43c10000, offset=0x1e70)[1])

In [ ]:
20102/16/4096*3.33

In [ ]:
o1.set_logging_level('debug')

In [ ]:
# switch pl
o1.register_read(base_address=0x43c10000, offset=0x193c)

In [31]:
o1.register_read(base_address=0x43c10000, offset=0x1940)

(0, b'\x00\x00\x00\x00')

In [295]:
o1.register_write(base_address=0x43c10000, offset=0x1940, data=0x0000_0000)

(0, None)

In [190]:
o1.register_write(base_address=0x43c10000, offset=0x1940, data=0xffff_ffff)

(0, None)

In [ ]:
o1.register_read(base_address=0x43c10000, offset=0x1940)

In [204]:
o1.register_write(base_address=0x43c10000, offset=0x1944, data=0xffff_ffff)

(0, None)

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x1944, data=0b0000_0000_0000_0000_0100_0000_0000)

In [ ]:
import time
for i in range(32):
    print(i)
    o1.register_write(base_address=0x43c10000, offset=0x1944, data=1 << i)
    time.sleep(1)

In [203]:
o1.register_write(base_address=0x43c10000, offset=0x1944, data=0x0000_0000)

(0, None)

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x1944, data=0b10000000)

In [ ]:
o1.register_read(base_address=0x43c10000, offset=0x1944)

In [301]:
o1.register_write(base_address=0x43c10000, offset=0x194c, data=0x0000_0000)

(0, None)

In [ ]:
# gpio输入模式测试

In [311]:
o1.register_write(base_address=0x43c10000, offset=0x1940, data=0xffff_ffff)

(0, None)

In [526]:
[f'{byte:08b}' for byte in o1.register_read(base_address=0x43c10000, offset=0x1948)[1]]

['00000000', '00001111', '00000111', '11111100']

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x194c, data=0xffff_ffff)

In [305]:
[f'{byte:08b}' for byte in o1.register_read(base_address=0x43c10000, offset=0x1954)[1]]

['00000000', '00000000', '00000000', '00000000']

In [429]:
o1.digital_read('io4')

(0, 1)

In [ ]:
# gpio 输入模式测试止

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x1950, data=0xffff_ffff)

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x1950, data=0b1)

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x1950, data=0b0000_0000_0000_0000_0000_0100_0000_0000)

In [ ]:
o1.register_write(base_address=0x43c10000, offset=0x1950, data=0b100_0000)

In [ ]:
o1.get_switch_status('pl')

In [ ]:
o1.set_led_text('HHHHHHHHHHH', y = 6)

In [ ]:
o1.set_led_text('HHHHHHHHHHHasdfasdfs', y = 6, x = 10)

In [5]:
o1.digital_write('a2', 1)

the old is 00000400
the new is 00080400


(0, None)

In [6]:
o1.digital_pin_mode('a', 1)

(0, None)

In [8]:
o1.digital_read('a')

(0, 1)

In [ ]:
# 配置与读取 数字接口
o1.digital_pin_mode('a', 1) # 配置为输入模式
o1.digital_read('a') # 读取
o1.digital_pin_mode('a', 0) # 配置为输出模式
o1.digital_write('a2', 1) # 设置高电平

In [9]:
# 测试所有gpio的读取
import time
from IPython.display import clear_output, display, Markdown

pin_ids = ["GP7","GP0","GP1","GP2","GP3","GP4","GP5","GP6","GP21","GP22","GP26","GP23","GP24","GP27","GP25","A2","A3","A4","A5","IO2","IO3","IO4","IO5","IO6","IO7","IO8","IO9","A"]

for pin_id in pin_ids:
    o1.digital_pin_mode(pin_id, 1)

for _ in range(100):
    clear_output(wait=True)
    vs = []
    for pin_id in pin_ids:
        r, v = o1.digital_read(pin_id)
        vs.append((pin_id, v))
    
    print(*vs, sep='\n')
    time.sleep(0.5)
print("完成！")

正在处理第 100 组数据... 进度: 100%
完成！
